In [2]:
import os
import timeit

import numpy as np
import pandas as pd
from IPython import get_ipython

import pyvinecopulib as pv

ipython = get_ipython()

# Display plots inline in Jupyter Notebook
ipython.run_line_magic("matplotlib", "inline")

# Minimal reporting for exception handlers
ipython.run_line_magic("xmode", "minimal")

# Automatically reload modules when they have changed
ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

# Display all output from a cell (not just the last line)
ipython.run_line_magic(
  "config", "InteractiveShell.ast_node_interactivity='all'"
)

# Set display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("max_colwidth", None)
pd.set_option("display.precision", 3)

# Numpy settings
np.printoptions(
  precision=4,
  suppress=True,
  formatter={"float": "{:0.4f}".format},
  linewidth=80,
)

Exception reporting mode: Minimal


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from vinesforest.margin_threshold_selector import MarginThresholdSelector
from vinesforest.vines_forest import VinesForest

n_vines = 100
seed = 12
target_precision = 0.95  # adjust to taste
cv_splits = 5
num_threads = min(32, os.cpu_count() + 4)

data = pd.read_csv("data/uranium.csv")
if (data.values < 0).any() or (data.values > 1).any():
  data[data.columns] = pv.to_pseudo_obs(data.values)

controls = pv.FitControlsVinecop(family_set=[pv.tll], num_threads=num_threads)

data_train, data_test = train_test_split(data, test_size=0.5, random_state=seed)

# --- Dissmann baseline ---
start = timeit.default_timer()
fitted_dissmann = pv.Vinecop.from_data(data_train.values, controls=controls)
# Per-observation logliks
ll_diss_train_obs = fitted_dissmann.loglik(
  data_train.values, per_observation=True
)  # shape (1, n_train) or (n_train,)
ll_diss_test_obs = fitted_dissmann.loglik(
  data_test.values, per_observation=True
)  # shape (1, n_test)  or (n_test,)

# Ensure shape (1, n_obs)
ll_diss_train_obs = np.atleast_2d(ll_diss_train_obs)
ll_diss_test_obs = np.atleast_2d(ll_diss_test_obs)
time_dissmann = timeit.default_timer() - start

# --- Forest alternatives ---
start = timeit.default_timer()
fitted_forest = VinesForest(n_vines=n_vines, controls=controls)
fitted_forest.fit(data_train.values)

# Per-observation logliks for each of the n_vines
# Expected shape: (n_vines, n_obs)
ll_forest_train_obs = fitted_forest.loglik(
  data_train.values, per_observation=True
)
ll_forest_test_obs = fitted_forest.loglik(
  data_test.values, per_observation=True
)
time_forest = timeit.default_timer() - start

assert ll_forest_train_obs.shape[0] == n_vines
assert ll_forest_test_obs.shape[0] == n_vines

# ---------------------------
# Method 1: pick τ via CV to hit target precision
# ---------------------------
selector = MarginThresholdSelector(
  cv=cv_splits, target_precision=target_precision, n_tau=200, random_state=seed
).fit(
  X_alt_train=ll_forest_train_obs,
  X_base_train=np.repeat(ll_diss_train_obs, n_vines, axis=0),
  X_alt_test=ll_forest_test_obs,
  X_base_test=np.repeat(ll_diss_test_obs, n_vines, axis=0),
)

y_pred = selector.decision_from_train(
  X_alt_train=ll_forest_train_obs,
  X_base_train=np.repeat(ll_diss_train_obs, n_vines, axis=0),
)

eval_test = selector.evaluate_on_test(
  y_pred_from_train=y_pred,
  X_alt_test=ll_forest_test_obs,
  X_base_test=np.repeat(ll_diss_test_obs, n_vines, axis=0),
)

print(f"Selected tau (ΔLL per-observation mean): {selector.tau_:.6f}")
print(pd.DataFrame([eval_test], index=["Test Metrics"]))

# Optional: CV operating curve (precision vs tau)
# print(selector.cv_summary_.sort_values("tau").tail(10))

# ---------------------------
# Baseline summaries (for reference)
# ---------------------------
logliks = [
  ["Dissmann", "Train", float(ll_diss_train_obs.mean())],
  ["Dissmann", "Test", float(ll_diss_test_obs.mean())],
  ["Forest (best m)", "Train", float(ll_forest_train_obs.mean(axis=1).max())],
  ["Forest (best m)", "Test", float(ll_forest_test_obs.mean(axis=1).max())],
]
df_logliks = pd.DataFrame(
  logliks, columns=["Method", "Set", "Mean per-obs Loglik"]
)
print(df_logliks)

print(f"Time Dissmann: {time_dissmann:.3f}s")
print(f"Time Forest:   {time_forest:.3f}s")

TypeError: loglik(): incompatible function arguments. The following argument types are supported:
    1. loglik(self, u: numpy.ndarray[dtype=float64, shape=(*, *), order='F'] = array([], shape=(0, 0), dtype=float64), num_threads: int = 1) -> float

Invoked with types: pyvinecopulib.Vinecop, ndarray, kwargs = { per_observation: bool }